In [1]:
import os
import pandas as pd
from pathlib import Path
from glob import glob
from neuron_morphology.swc_io import morphology_from_swc
from neuron_morphology.features.size import total_length

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# Calculate Total Apical and Basal Dendrite Length

This notebook calculates total apical (node_type=4) and basal (node_type=3) dendrite lengths for morphologies across human, macaque, and mouse species using raw SWC files.

In [2]:
# Define paths to raw SWC directories
DATA_ROOT = Path('/root/capsule/data')

swc_dirs = {
    'human': DATA_ROOT / 'human' / 'morphology' / 'swc' / '20260625' / 'SWC_Raw',
    'macaque': DATA_ROOT / 'macaque' / 'morphology' / 'swc' / '20260722' / 'SWC_Raw',
    'mouse': DATA_ROOT / 'mouse' / 'morphology' / 'swc' / '20260826' / 'SWC_Raw'
}

# Output directory
OUT_DIR = Path('/root/capsule/scratch/20260807/species_comparison')
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
def calculate_dendrite_lengths(swc_path):
    """
    Calculate total apical and basal dendrite lengths from an SWC file.
    
    Parameters:
    -----------
    swc_path : str or Path
        Path to the SWC file
    
    Returns:
    --------
    tuple : (apical_length, basal_length)
        Total apical dendrite length (node_type=4) and total basal dendrite length (node_type=3)
    """
    try:
        # Load morphology
        morphology = morphology_from_swc(swc_path)
        
        # Calculate apical dendrite length (node_type=4)
        apical_length = total_length(morphology, node_types=[4])
        
        # Calculate basal dendrite length (node_type=3)
        basal_length = total_length(morphology, node_types=[3])
        
        return apical_length, basal_length
    
    except Exception as e:
        print(f"Error processing {swc_path}: {e}")
        return None, None

In [4]:
# Collect results for all species
results = []

for species, swc_dir in swc_dirs.items():
    print(f"\nProcessing {species} morphologies from {swc_dir}...")
    
    # Get all SWC files
    swc_files = sorted(glob(str(swc_dir / '*.swc')))
    print(f"Found {len(swc_files)} SWC files")
    
    for swc_path in swc_files:
        # Extract specimen_id (filename without .swc extension)
        specimen_id = Path(swc_path).stem
        
        # Calculate dendrite lengths
        apical_length, basal_length = calculate_dendrite_lengths(swc_path)
        
        if apical_length is not None:
            results.append({
                'specimen_id': specimen_id,
                'species': species,
                'apical_length': apical_length,
                'basal_length': basal_length
            })

print(f"\nTotal morphologies processed: {len(results)}")


Processing human morphologies from /root/capsule/data/human/morphology/swc/20260625/SWC_Raw...
Found 368 SWC files

Processing macaque morphologies from /root/capsule/data/macaque/morphology/swc/20260722/SWC_Raw...
Found 51 SWC files

Processing mouse morphologies from /root/capsule/data/mouse/morphology/swc/20260826/SWC_Raw...
Found 29 SWC files

Total morphologies processed: 448


In [5]:
# Create DataFrame
df_results = pd.DataFrame(results)
df_results.head(10)

,specimen_id,species,apical_length,basal_length
0,1001449781,human,3516.704254,1874.560061
1,1001462085,human,5378.076521,6350.964035
2,1001491745,human,2210.589125,1330.410677
3,1001497435,human,1243.799101,1430.368616
4,1005032096,human,2149.147827,1266.004051
5,1005032781,human,1105.917878,515.817291
6,1005040969,human,1419.302590,1689.787271
7,1005071140,human,2416.339909,2560.955535
8,1009206585,human,2172.649702,1442.836391
9,1010237164,human,1259.519719,816.175533


In [6]:
# Summary statistics by species
print("\nSummary statistics by species:")
print("\nSample counts:")
print(df_results.groupby('species').size())

print("\nApical length (mean ± std):")
print(df_results.groupby('species')['apical_length'].agg(['mean', 'std']))

print("\nBasal length (mean ± std):")
print(df_results.groupby('species')['basal_length'].agg(['mean', 'std']))


Summary statistics by species:

Sample counts:
species
human      368
macaque     51
mouse       29
dtype: int64

Apical length (mean ± std):
                mean          std
species                          
human    3850.461071  2141.158373
macaque  6669.369713  3573.528756
mouse    2270.525603   950.368109

Basal length (mean ± std):
                mean          std
species                          
human    3218.225207  1894.708285
macaque  4767.612717  1540.960836
mouse    2638.541032   727.539359


In [7]:
# Save to CSV
output_file = OUT_DIR / 'species_dendrite_lengths_raw.csv'
df_results.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")


Results saved to: /root/capsule/scratch/20260807/species_comparison/species_dendrite_lengths_raw.csv
